In [1]:
# Imports
import sys
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

from tqdm import tqdm


# Project Imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils

In [4]:
# =========================================================
# Input Files
# =========================================================

PATH_RESULTS = (
    cfg.PROCESSED_DIR / "final_dataset.parquet"
)

df_final = pd.read_parquet(PATH_RESULTS)

PATH_BASELINE = (
    cfg.DATA_DIR / "metadata/baseline.csv"
)
df_baseline = pd.read_csv(PATH_BASELINE)

In [5]:
# Combined dataset containing all six solvers and their raw runs
df_runs = df_final.copy()

solvers = [
    "DAv2",
    "DAv3",
    "DAv3c",
    "BURER2002",
    "MERZ1999GLS",
    "PALUBECKIS2004bMST2"
]

# Exact primary-instance selection
primary_names = set(
    df_baseline.loc[
        (df_baseline["nodes"] >= 2048)
        & (df_baseline["limit"] > 0.25),
        "name"
    ]
)

assert len(primary_names) == 819, (
    f"Expected 819 primary instances, found {len(primary_names)}"
)

df_primary = df_runs[
    df_runs["name"].isin(primary_names)
    & df_runs["heuristic"].isin(solvers)
].copy()

# Use baseline reference limits, rather than relying
# on whichever limit column the merge notebook produced.
if "limit" in df_primary.columns:
    df_primary = df_primary.drop(columns=["limit"])

df_primary = df_primary.merge(
    df_baseline[["name", "limit"]].drop_duplicates("name"),
    on="name",
    how="left",
    validate="many_to_one"
)

assert df_primary["limit"].notna().all()
assert (df_primary["limit"] > 0).all()
assert df_primary["runtime"].notna().all()

df_primary["runtime_ratio"] = (
    df_primary["runtime"] / df_primary["limit"]
)

df_primary["excess_pct"] = (
    100 * (df_primary["runtime_ratio"] - 1)
)

print("Primary instances:", df_primary["name"].nunique())
print("Total raw runs:", len(df_primary))

display(
    df_primary.groupby("heuristic").agg(
        instances=("name", "nunique"),
        runs=("name", "size")
    ).reindex(solvers)
)

Primary instances: 819
Total raw runs: 24145


,instances,runs
heuristic,,
DAv2,738,3690
DAv3,819,4095
DAv3c,819,4095
BURER2002,815,4075
MERZ1999GLS,819,4095
PALUBECKIS2004bMST2,819,4095


In [6]:
run_check = (
    df_primary.groupby(["name", "heuristic"])
    .agg(
        runs=("seed", "size"),
        unique_seeds=("seed", "nunique")
    )
)

problems = run_check[
    (run_check["runs"] != 5)
    | (run_check["unique_seeds"] != 5)
]

print("Invalid solver-instance groups:", len(problems))
display(problems)

assert problems.empty, "Investigate invalid run groups first."

Invalid solver-instance groups: 0


,,runs,unique_seeds
name,heuristic,,


In [7]:
def runtime_stats(df):
    return pd.Series({
        "instances": df["name"].nunique(),
        "runs": len(df),
        "over_5pct": (df["runtime_ratio"] > 1.05).sum(),
        "over_10pct": (df["runtime_ratio"] > 1.10).sum(),
        "over_2x": (df["runtime_ratio"] > 2).sum(),
        "median_ratio": df["runtime_ratio"].median(),
        "p95_ratio": df["runtime_ratio"].quantile(0.95),
        "max_ratio": df["runtime_ratio"].max()
    })

raw_runtime_summary = (
    df_primary
    .groupby("heuristic")
    .apply(runtime_stats, include_groups=False)
    .reindex(solvers)
)

for threshold in ["over_5pct", "over_10pct", "over_2x"]:
    raw_runtime_summary[threshold + "_pct"] = (
        100 * raw_runtime_summary[threshold]
        / raw_runtime_summary["runs"]
    )

display(raw_runtime_summary.round(4))

,instances,runs,over_5pct,over_10pct,over_2x,median_ratio,p95_ratio,max_ratio,over_5pct_pct,over_10pct_pct,over_2x_pct
heuristic,,,,,,,,,,,
DAv2,738.0,3690.0,0.0,0.0,0.0,0.9896,0.9965,1.0118,0.0000,0.0000,0.0
DAv3,819.0,4095.0,383.0,38.0,0.0,1.0080,1.0651,1.5046,9.3529,0.9280,0.0
DAv3c,819.0,4095.0,0.0,0.0,0.0,0.9950,1.0052,1.0206,0.0000,0.0000,0.0
BURER2002,815.0,4075.0,15.0,5.0,0.0,1.0016,1.0195,1.1462,0.3681,0.1227,0.0
MERZ1999GLS,819.0,4095.0,0.0,0.0,0.0,1.0002,1.0012,1.0104,0.0000,0.0000,0.0
PALUBECKIS2004bMST2,819.0,4095.0,0.0,0.0,0.0,1.0031,1.0174,1.0396,0.0000,0.0000,0.0


In [8]:
df_best = (
    df_primary
    .sort_values(
        ["name", "heuristic", "objective", "runtime"],
        ascending=[True, True, False, True]
    )
    .drop_duplicates(["name", "heuristic"])
    .copy()
)

best_runtime_summary = (
    df_best
    .groupby("heuristic")
    .apply(runtime_stats, include_groups=False)
    .reindex(solvers)
)

for threshold in ["over_5pct", "over_10pct", "over_2x"]:
    best_runtime_summary[threshold + "_pct"] = (
        100 * best_runtime_summary[threshold]
        / best_runtime_summary["runs"]
    )

display(best_runtime_summary.round(4))

,instances,runs,over_5pct,over_10pct,over_2x,median_ratio,p95_ratio,max_ratio,over_5pct_pct,over_10pct_pct,over_2x_pct
heuristic,,,,,,,,,,,
DAv2,738.0,738.0,0.0,0.0,0.0,0.9893,0.9961,1.0118,0.0000,0.0000,0.0
DAv3,819.0,819.0,68.0,8.0,0.0,1.0051,1.0620,1.5046,8.3028,0.9768,0.0
DAv3c,819.0,819.0,0.0,0.0,0.0,0.9948,1.0050,1.0141,0.0000,0.0000,0.0
BURER2002,815.0,815.0,2.0,0.0,0.0,1.0013,1.0194,1.0637,0.2454,0.0000,0.0
MERZ1999GLS,819.0,819.0,0.0,0.0,0.0,1.0001,1.0008,1.0036,0.0000,0.0000,0.0
PALUBECKIS2004bMST2,819.0,819.0,0.0,0.0,0.0,1.0026,1.0169,1.0362,0.0000,0.0000,0.0


In [9]:
dav3_best = df_best[
    df_best["heuristic"] == "DAv3"
]

print("DAv3 best-of-five records:", len(dav3_best))
print("Over 5%:", (dav3_best["runtime_ratio"] > 1.05).sum())
print("Over 10%:", (dav3_best["runtime_ratio"] > 1.10).sum())
print("Over 2x:", (dav3_best["runtime_ratio"] > 2).sum())
print("Median ratio:", dav3_best["runtime_ratio"].median())
print("95th percentile:", dav3_best["runtime_ratio"].quantile(0.95))
print("Maximum ratio:", dav3_best["runtime_ratio"].max())

DAv3 best-of-five records: 819
Over 5%: 68
Over 10%: 8
Over 2x: 0
Median ratio: 1.0051144999517378
95th percentile: 1.0620172339095593
Maximum ratio: 1.5046337357212005


In [10]:
anomalies = (
    df_best[
        df_best["runtime_ratio"] > 1.10
    ]
    .sort_values(
        "runtime_ratio",
        ascending=False
    )
    .copy()
)

display(
    anomalies[
        [
            "name",
            "heuristic",
            "limit",
            "runtime",
            "runtime_ratio",
            "excess_pct",
            "objective",
            "seed",
            "file"
        ]
    ].round(4)
)

,name,heuristic,limit,runtime,runtime_ratio,excess_pct,objective,seed,file
9136,g000967,DAv3,2.066,3.1086,1.5046,50.4634,5.180000e+02,244,./DAv3/g000967-1-5.csv
11244,g003215,DAv3,2.136,2.9925,1.4010,40.0990,8.188060e+05,544,./DAv3/g003215-1-5.csv
10309,g002242,DAv3,2.609,3.0014,1.1504,15.0409,4.320000e+03,544,./DAv3/g002242-1-5.csv
7800,G62,DAv3,22.082,25.3582,1.1484,14.8364,4.854000e+03,144,./DAv3/G62-21-5.csv
7819,G65,DAv3,28.665,32.4572,1.1323,13.2294,5.544000e+03,544,./DAv3/G65-28-5.csv
11386,imgseg_163014,DAv3,30.175,33.3967,1.1068,10.6767,7.895940e+06,244,./DAv3/imgseg_163014-30-5.csv
10343,g002276,DAv3,32.346,35.7118,1.1041,10.4056,7.968600e+04,444,./DAv3/g002276-32-5.csv
10753,g002775,DAv3,25.011,27.5501,1.1015,10.1520,5.921470e+05,444,./DAv3/g002775-24-5.csv


In [11]:
extreme = df_best[
    df_best["runtime_ratio"] > 2
].copy()

print("Extreme records:", len(extreme))
print("Affected instances:", extreme["name"].nunique())

display(
    extreme[
        [
            "name",
            "heuristic",
            "limit",
            "runtime",
            "runtime_ratio",
            "file"
        ]
    ].sort_values(
        "runtime_ratio",
        ascending=False
    )
)

Extreme records: 0
Affected instances: 0


,name,heuristic,limit,runtime,runtime_ratio,file


In [12]:
reviewer_instances = [
    "g003215", "g000989", "g001269", "g002569",
    "g000377", "g001086", "g002440", "g001337",
    "g001883", "g000283", "imgseg_376020",
    "imgseg_216041", "g000476"
]

reviewer_check = df_best[
    df_best["name"].isin(reviewer_instances)
    & (df_best["heuristic"] == "BURER2002")
].copy()

display(
    reviewer_check[
        [
            "name",
            "limit",
            "runtime",
            "runtime_ratio",
            "file"
        ]
    ].sort_values("name").round(4)
)

assert (reviewer_check["runtime_ratio"] <= 2).all()

,name,limit,runtime,runtime_ratio,file
845,g000283,36.585,36.6755,1.0025,./BURER2002/g000283-36.585-5.csv
891,g000377,7.008,7.0173,1.0013,./BURER2002/g000377-7.008-5.csv
989,g000476,57.363,57.4093,1.0008,./BURER2002/g000476-57.363-5.csv
1412,g000989,2.874,2.8769,1.0010,./BURER2002/g000989-2.874-5.csv
1516,g001086,8.818,8.8274,1.0011,./BURER2002/g001086-8.818-5.csv
1689,g001269,5.084,5.0906,1.0013,./BURER2002/g001269-5.084-5.csv
1750,g001337,18.048,18.2677,1.0122,./BURER2002/g001337-18.048-5.csv
2161,g001883,24.803,24.8105,1.0003,./BURER2002/g001883-24.803-5.csv
2708,g002440,11.756,11.7619,1.0005,./BURER2002/g002440-11.756-5.csv
2822,g002569,5.388,5.3916,1.0007,./BURER2002/g002569-5.388-5.csv


In [13]:
df_five_run_cost = (
    df_primary.groupby(["name", "heuristic"])
    .agg(
        assigned_tl=("limit", "first"),
        total_runtime_5runs=("runtime", "sum"),
        mean_runtime=("runtime", "mean"),
        min_runtime=("runtime", "min"),
        max_runtime=("runtime", "max"),
        n_runs=("runtime", "size")
    )
    .reset_index()
)

# Join the best-of-five record
df_five_run_cost = df_five_run_cost.merge(
    df_best[
        ["name", "heuristic", "objective", "runtime"]
    ].rename(columns={
        "objective": "best_objective",
        "runtime": "selected_runtime"
    }),
    on=["name", "heuristic"],
    how="left",
    validate="one_to_one"
)

# Total recorded effort compared with one reference TL
df_five_run_cost["total_cost_ratio"] = (
    df_five_run_cost["total_runtime_5runs"]
    / df_five_run_cost["assigned_tl"]
)

# Total recorded effort compared with five reference TLs
df_five_run_cost["five_budget_ratio"] = (
    df_five_run_cost["total_runtime_5runs"]
    / (5 * df_five_run_cost["assigned_tl"])
)

display(df_five_run_cost.head())

,name,heuristic,assigned_tl,total_runtime_5runs,mean_runtime,min_runtime,max_runtime,n_runs,best_objective,selected_runtime,total_cost_ratio,five_budget_ratio
0,G55,BURER2002,13.945,69.805499,13.961100,13.946340,13.970771,5,10249.0,13.960023,5.005773,1.001155
1,G55,DAv2,13.945,68.945641,13.789128,13.765187,13.833018,5,10294.0,13.773063,4.944112,0.988822
2,G55,DAv3,13.945,62.855210,12.571042,12.473567,12.691473,5,10294.0,12.691473,4.507365,0.901473
3,G55,DAv3c,13.945,65.597121,13.119424,13.118277,13.121273,5,10200.0,13.118277,4.703989,0.940798
4,G55,MERZ1999GLS,13.945,69.735377,13.947075,13.945797,13.948499,5,10169.0,13.945797,5.000744,1.000149


In [14]:
cost_summary = (
    df_five_run_cost.groupby("heuristic")
    .agg(
        instances=("name", "nunique"),
        total_runtime_hours=(
            "total_runtime_5runs",
            lambda x: x.sum() / 3600
        ),
        total_selected_hours=(
            "selected_runtime",
            lambda x: x.sum() / 3600
        ),
        median_total_cost_ratio=(
            "total_cost_ratio", "median"
        ),
        median_five_budget_ratio=(
            "five_budget_ratio", "median"
        )
    )
    .reindex(solvers)
)

cost_summary["total_vs_selected"] = (
    cost_summary["total_runtime_hours"]
    / cost_summary["total_selected_hours"]
)

display(cost_summary.round(3))

,instances,total_runtime_hours,total_selected_hours,median_total_cost_ratio,median_five_budget_ratio,total_vs_selected
heuristic,,,,,,
DAv2,738,100.754,20.148,4.949,0.990,5.001
DAv3,819,156.987,31.362,5.042,1.008,5.006
DAv3c,819,153.828,30.764,4.974,0.995,5.000
BURER2002,815,134.975,26.995,5.010,1.002,5.000
MERZ1999GLS,819,155.401,31.072,5.001,1.000,5.001
PALUBECKIS2004bMST2,819,155.653,31.130,5.018,1.004,5.000


In [15]:
dav2_names = set(
    df_best.loc[
        df_best["heuristic"] == "DAv2",
        "name"
    ]
)

df_shared_cost = df_five_run_cost[
    df_five_run_cost["name"].isin(dav2_names)
].copy()

shared_summary = (
    df_shared_cost.groupby("heuristic")
    .agg(
        instances=("name", "nunique"),
        total_five_run_hours=(
            "total_runtime_5runs",
            lambda x: x.sum() / 3600
        ),
        median_total_cost_ratio=(
            "total_cost_ratio", "median"
        )
    )
    .reindex(solvers)
)

display(shared_summary.round(3))

,instances,total_five_run_hours,median_total_cost_ratio
heuristic,,,
DAv2,738,100.754,4.949
DAv3,738,102.425,5.034
DAv3c,738,101.541,4.965
BURER2002,738,102.766,5.011
MERZ1999GLS,738,101.846,5.001
PALUBECKIS2004bMST2,738,101.982,5.017


In [16]:
coverage = (
    df_five_run_cost.groupby("name")["heuristic"]
    .nunique()
)

common_names = coverage[coverage == len(solvers)].index

print("Instances with all six solvers:", len(common_names))

df_common_cost = df_five_run_cost[
    df_five_run_cost["name"].isin(common_names)
].copy()

display(
    df_common_cost.groupby("heuristic").agg(
        instances=("name", "nunique"),
        total_five_run_hours=(
            "total_runtime_5runs",
            lambda x: x.sum() / 3600
        ),
        median_cost_ratio=("total_cost_ratio", "median")
    ).reindex(solvers).round(3)
)

Instances with all six solvers: 738


,instances,total_five_run_hours,median_cost_ratio
heuristic,,,
DAv2,738,100.754,4.949
DAv3,738,102.425,5.034
DAv3c,738,101.541,4.965
BURER2002,738,102.766,5.011
MERZ1999GLS,738,101.846,5.001
PALUBECKIS2004bMST2,738,101.982,5.017


In [17]:
from pathlib import Path

audit_dir = Path("runtime_audit")
audit_dir.mkdir(parents=True, exist_ok=True)

raw_runtime_summary.to_csv(
    audit_dir / "runtime_all_trials.csv"
)

best_runtime_summary.to_csv(
    audit_dir / "runtime_best_of_five.csv"
)

df_best.to_csv(
    audit_dir / "selected_best_runs.csv",
    index=False
)

anomalies.to_csv(
    audit_dir / "runtime_overruns_10pct.csv",
    index=False
)

df_five_run_cost.to_csv(
    audit_dir / "five_run_cost_by_instance.csv",
    index=False
)

cost_summary.to_csv(
    audit_dir / "five_run_cost_summary.csv"
)

print("Saved runtime audit outputs to:", audit_dir.resolve())

Saved runtime audit outputs to: /Users/sargun/Desktop/DAMB-DAv3c/notebooks/runtime_audit
